#### Enigma Part

* basically we need it to encrypt our cribs
* which means that it encrypts the cribs and then checks back with the bombe
* keeps doing this so that it can find whats right and whats wrong
* without this part its like having a key but no lock to put it in

In [1]:
import time
import string
from typing import List, Tuple, Optional, Dict, Union
from itertools import product
from datetime import datetime
import os


class EnigmaMachine:
    """3-rotor Enigma machine implementation"""
    
    ROTORS = {
        'I':   'EKMFLGDQVZNTOWYHXUSPAIBRCJ',
        'II':  'AJDKSIRUXBLHWTMCQGZNPYFVOE',
        'III': 'BDFHJLCPRTXVZNYEIWGAKMUSQO',
        'IV':  'ESOVPZJAYQUIRHXLNFTGKDCMWB',
        'V':   'VZBRGITYUPSDNHLXAWMJQOFECK'
    }
    
    NOTCHES = {'I': 'Q', 'II': 'E', 'III': 'V', 'IV': 'J', 'V': 'Z'}
    REFLECTOR_B = 'YRUHQSLDPXNGOKMIEBFZCWVJAT'
    ALPHABET = string.ascii_uppercase
    
    def __init__(self, rotors: Tuple[str, str, str], positions: Tuple[int, int, int],
                 ring_settings: Tuple[int, int, int] = (0, 0, 0),
                 plugboard: Dict[str, str] = None):
        self.rotors = [self.ROTORS[r] for r in rotors]
        self.rotor_names = rotors
        self.positions = list(positions)
        self.ring_settings = list(ring_settings)
        self.notches = [self.NOTCHES[r] for r in rotors]
        self.reflector = self.REFLECTOR_B
        self.plugboard = plugboard or {}
    
    def _plugboard_swap(self, char: str) -> str:
        return self.plugboard.get(char, char)
    
    def _rotate_rotors(self):
        if self.ALPHABET[self.positions[1]] == self.notches[1]:
            self.positions[1] = (self.positions[1] + 1) % 26
            self.positions[2] = (self.positions[2] + 1) % 26
        elif self.ALPHABET[self.positions[0]] == self.notches[0]:
            self.positions[1] = (self.positions[1] + 1) % 26
        self.positions[0] = (self.positions[0] + 1) % 26
    
    def _encode_through_rotor(self, char_index: int, rotor_index: int, forward: bool = True) -> int:
        rotor = self.rotors[rotor_index]
        position = self.positions[rotor_index]
        ring = self.ring_settings[rotor_index]
        
        if forward:
            shifted = (char_index + position - ring) % 26
            encoded = self.ALPHABET.index(rotor[shifted])
            return (encoded - position + ring) % 26
        else:
            shifted = (char_index + position - ring) % 26
            encoded = rotor.index(self.ALPHABET[shifted])
            return (encoded - position + ring) % 26
    
    def _encode_through_reflector(self, char_index: int) -> int:
        return self.ALPHABET.index(self.reflector[char_index])
    
    def encrypt_char(self, char: str) -> str:
        if char not in self.ALPHABET:
            return char
        
        self._rotate_rotors()
        char = self._plugboard_swap(char)
        char_index = self.ALPHABET.index(char)
        
        for i in range(3):
            char_index = self._encode_through_rotor(char_index, i, forward=True)
        
        char_index = self._encode_through_reflector(char_index)
        
        for i in range(2, -1, -1):
            char_index = self._encode_through_rotor(char_index, i, forward=False)
        
        result = self.ALPHABET[char_index]
        result = self._plugboard_swap(result)
        return result
    
    def encrypt(self, text: str) -> str:
        return ''.join(self.encrypt_char(c.upper()) for c in text if c.isalpha())




In [2]:
# ============================================================
# IOC VERIFICATION + INITIAL POSITION RECOVERY
# ============================================================

def index_of_coincidence(text):
    """
    Index of Coincidence (IoC):
    Real English = ~0.065 | Random/encrypted = ~0.038
    """
    n = len(text)
    if n < 2:
        return 0
    counts = {}
    for c in text:
        counts[c] = counts.get(c, 0) + 1
    return sum(f * (f - 1) for f in counts.values()) / (n * (n - 1))


def find_initial_positions(rotors, positions_at_crib, crib_text_position):
    """
    THE KEY FIX:
    The Bombe finds rotor positions AT the crib's location in the text.
    But to decrypt the FULL message we need the INITIAL positions (at char 0).

    Example:
      Text encrypted starting at CZT (2,25,19)
      After 21 characters, rotors are at XAT (23,0,19)
      Bombe correctly finds XAT for the crib at position 21
      This function works backwards from XAT to recover CZT

    Strategy: try all 17576 starting positions, simulate stepping
    through crib_text_position characters, check if we reach
    positions_at_crib.
    """
    for p0 in range(26):
        for p1 in range(26):
            for p2 in range(26):
                e = EnigmaMachine(rotors, (p0, p1, p2))
                # Simulate exactly crib_text_position rotor steps
                for _ in range(crib_text_position):
                    e._rotate_rotors()
                # Did we land at the position the Bombe found?
                if tuple(e.positions) == tuple(positions_at_crib):
                    return (p0, p1, p2)
    return None


def is_valid_decryption(ciphertext, rotors, initial_positions):
    """
    Decrypt full ciphertext with INITIAL positions and check IoC.
    """
    enigma = EnigmaMachine(rotors, initial_positions)
    full_decrypt = enigma.encrypt(ciphertext)
    ioc = index_of_coincidence(full_decrypt)
    is_english = ioc >= 0.055
    return is_english, ioc, full_decrypt


#### Bombe Decoding Part
* New improvements
* has a sliding crib function which was the last ones main issue,
* basically it goes over each individual character up until it reaches a certain point
* did 500 but that sometimes takes way too long so can modify by either removing cribs
* or by increaseing the length of where it "slides" too

In [3]:
class Bombe:
    """
    Bombe with SLIDING CRIB SEARCH + IOC VERIFICATION
    Correctly recovers initial rotor positions from crib position.
    """

    MIN_CRIB_LENGTH = 8

    def __init__(self, rotors_to_test=None):
        if rotors_to_test is None:
            self.rotors_to_test = [
                ('I', 'II', 'III'),
                ('I', 'III', 'II'),
                ('II', 'I', 'III'),
                ('II', 'III', 'I'),
                ('III', 'I', 'II'),
                ('III', 'II', 'I')
            ]
        else:
            self.rotors_to_test = rotors_to_test

    def _test_crib(self, ciphertext_slice, crib, rotors, positions, plugboard=None):
        """Test if encrypting crib with these settings matches the ciphertext slice"""
        try:
            enigma = EnigmaMachine(rotors, positions, plugboard=plugboard)
            return enigma.encrypt(crib) == ciphertext_slice
        except:
            return False

    def break_with_sliding_cribs(self, ciphertext, cribs,
                                  max_positions=17576,
                                  max_slide_distance=200,
                                  plugboard=None,
                                  verbose=True):
        """
        Search for cribs at ANY position in the ciphertext.
        When a crib match is found:
          1. Recovers the INITIAL rotor positions from the crib position
          2. Verifies by decrypting the full text and checking IoC
        """
        ciphertext = ''.join(c for c in ciphertext if c.isalpha()).upper()

        print(f"\n{'='*70}")
        print(f"SLIDING CRIB SEARCH WITH IOC VERIFICATION")
        print(f"{'='*70}")
        print(f"Cribs to try: {len(cribs)}")
        print(f"Searching first {max_slide_distance} characters")
        print(f"IoC threshold: 0.055  (English=0.065, Random=0.038)")
        print(f"{'='*70}\n")

        overall_start = time.time()
        total_tests = 0

        for crib_num, crib in enumerate(cribs, 1):
            crib = crib.upper().replace(' ', '')

            print(f"\n{'='*70}")
            print(f"CRIB {crib_num}/{len(cribs)}: '{crib}' ({len(crib)} letters)")
            print(f"{'='*70}")

            if len(crib) < self.MIN_CRIB_LENGTH:
                print(f"  WARNING: Too short ({len(crib)} < {self.MIN_CRIB_LENGTH}). Skipping!")
                continue

            search_limit = min(len(ciphertext) - len(crib), max_slide_distance)
            print(f"Searching positions 0 to {search_limit}...\n")

            for text_pos in range(search_limit + 1):
                cipher_slice = ciphertext[text_pos:text_pos + len(crib)]

                if verbose and text_pos % 20 == 0 and text_pos > 0:
                    print(f"  Checked {text_pos}/{search_limit} positions...", end='\r')

                for rotor_config in self.rotors_to_test:
                    for pos in product(range(26), range(26), range(26)):
                        total_tests += 1

                        if self._test_crib(cipher_slice, crib, rotor_config, pos, plugboard):

                            # ------------------------------------------------
                            # CRIB MATCHED at text_pos with rotor positions pos
                            # pos = positions AT text_pos (not initial positions!)
                            # We need to recover the INITIAL positions first
                            # ------------------------------------------------
                            print(f"\n  Crib match at text pos {text_pos}, "
                                  f"rotor state {pos} ({chr(65+pos[0])}{chr(65+pos[1])}{chr(65+pos[2])})")

                            if text_pos == 0:
                                # Crib is at start - pos IS the initial position
                                initial_pos = pos
                            else:
                                # Recover initial positions by working backwards
                                print(f"  Recovering initial positions from pos {text_pos}...", end='')
                                initial_pos = find_initial_positions(rotor_config, pos, text_pos)
                                if initial_pos is None:
                                    print(f" Could not recover. Skipping.")
                                    continue
                                print(f" Found: {initial_pos} ({chr(65+initial_pos[0])}{chr(65+initial_pos[1])}{chr(65+initial_pos[2])})")

                            # Now verify with full decryption using INITIAL positions
                            is_english, ioc, full_decrypt = is_valid_decryption(
                                ciphertext, rotor_config, initial_pos
                            )

                            print(f"  Verifying with initial pos {initial_pos} -> IoC: {ioc:.4f}", end='')

                            if is_english:
                                elapsed = time.time() - overall_start
                                print(f"  VERIFIED!")
                                print(f"\n{'='*70}")
                                print(f"SOLUTION FOUND AND VERIFIED!")
                                print(f"{'='*70}")
                                print(f"Crib '{crib}' at position:    {text_pos}")
                                print(f"Rotor config:                {rotor_config}")
                                print(f"Positions at crib:           {pos} ({chr(65+pos[0])}{chr(65+pos[1])}{chr(65+pos[2])})")
                                print(f"INITIAL positions (use this): {initial_pos} ({chr(65+initial_pos[0])}{chr(65+initial_pos[1])}{chr(65+initial_pos[2])})")
                                print(f"IoC Score:                   {ioc:.4f}  (English=0.065)")
                                print(f"Total tests:                 {total_tests:,}")
                                print(f"Time:                        {elapsed:.2f} seconds")
                                print(f"{'='*70}\n")

                                return {
                                    'success': True,
                                    'rotors': rotor_config,
                                    'positions': initial_pos,
                                    'positions_letters': f"{chr(65+initial_pos[0])}{chr(65+initial_pos[1])}{chr(65+initial_pos[2])}",
                                    'positions_at_crib': pos,
                                    'crib': crib,
                                    'crib_position': text_pos,
                                    'tests_performed': total_tests,
                                    'time_seconds': elapsed,
                                    'ioc': ioc,
                                    'successful_crib_number': crib_num,
                                    'total_cribs_tested': crib_num,
                                    'all_cribs': cribs,
                                    'crib_found_at_position': text_pos,
                                    'decrypted_text': full_decrypt
                                }
                            else:
                                print(f"  FALSE POSITIVE")

            print(f"\n No solution with '{crib}' in positions 0-{search_limit}")

        elapsed = time.time() - overall_start
        print(f"\n No solution found. Total tests: {total_tests:,}, Time: {elapsed:.2f}s")
        return {'success': False, 'message': 'No solution found'}


#### Main thing to run after setting cribs and searching length

In [4]:
# Helper functions (same as before)
def decrypt_message(ciphertext: str, rotors: Tuple[str, str, str], 
                   positions: Tuple[int, int, int],
                   plugboard: Dict[str, str] = None) -> str:
    enigma = EnigmaMachine(rotors, positions, plugboard=plugboard)
    return enigma.encrypt(ciphertext)


def calculate_accuracy(decrypted_text: str, expected_text: str) -> float:
    decrypted = decrypted_text.upper().replace(' ', '')
    expected = expected_text.upper().replace(' ', '')
    
    min_len = min(len(decrypted), len(expected))
    if min_len == 0:
        return 0.0
    
    matches = sum(1 for i in range(min_len) if decrypted[i] == expected[i])
    return (matches / min_len) * 100


def save_results(result, decrypted, ciphertext, original, accuracy, actual_settings, output_file):
    """Save results to file"""
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write("="*80 + "\n")
        f.write("BOMBE CRYPTANALYSIS RESULTS (SLIDING CRIB SEARCH)\n")
        f.write("="*80 + "\n\n")
        f.write(f"Timestamp: {timestamp}\n\n")
        
        f.write("-"*80 + "\n")
        f.write("SETTINGS FOUND BY BOMBE\n")
        f.write("-"*80 + "\n")
        f.write(f"Rotor Configuration: {result['rotors']}\n")
        f.write(f"Rotor Positions (numeric): {result['positions']}\n")
        f.write(f"Rotor Positions (letters): {result['positions_letters']}\n\n")
        
        if 'crib_found_at_position' in result:
            f.write(f"Crib '{result['crib']}' found at position: {result['crib_found_at_position']}\n\n")
        
        if actual_settings:
            f.write("-"*80 + "\n")
            f.write("ACTUAL ENCRYPTION SETTINGS\n")
            f.write("-"*80 + "\n")
            f.write(f"Rotor Configuration: {actual_settings['rotors']}\n")
            f.write(f"Rotor Positions (numeric): {actual_settings['positions']}\n")
            f.write(f"Rotor Positions (letters): {actual_settings['positions_letters']}\n\n")
            
            rotors_match = result['rotors'] == actual_settings['rotors']
            positions_match = result['positions'] == actual_settings['positions']
            
            f.write("VERIFICATION: ")
            f.write(f"{'✓ CORRECT' if (rotors_match and positions_match) else '✗ INCORRECT'}\n\n")
        
        f.write("-"*80 + "\n")
        f.write("PERFORMANCE\n")
        f.write("-"*80 + "\n")
        f.write(f"Time: {result['time_seconds']:.2f} seconds\n")
        f.write(f"Tests: {result['tests_performed']:,}\n")
        f.write(f"Accuracy: {accuracy:.2f}%\n\n")
        
        f.write("-"*80 + "\n")
        f.write("DECRYPTED TEXT (first 500 chars)\n")
        f.write("-"*80 + "\n")
        f.write(f"{decrypted[:500]}\n\n")
    
    print(f"✓ Results saved to: {output_file}")


def main():
    print("\n" + "="*80)
    print("BOMBE WITH SLIDING CRIB SEARCH")
    print("="*80 + "\n")
    
#####################################################################################################
    
    encrypted_file = r"C:\Users\tapia\Desktop\Python stuff\Classes\Capstone\training data\Encrypted Processed Discussion1.txt"      # Your encrypted file
    plaintext_file = r"C:\Users\tapia\Desktop\Python stuff\Classes\Capstone\training data\Processed Discussion1.txt"      # Original plaintext
    output_file = "bombe_results.txt"

#####################################################################################################
    actual_settings = {
        'rotors': ('I', 'II', 'III'),
        'positions': (2, 25, 19),
        'positions_letters': 'CZT'
    }
    
    # Read files
    if not os.path.exists(encrypted_file):
        print(f"ERROR: {encrypted_file} not found!")
        return
    
    with open(encrypted_file, 'r', encoding='utf-8') as f:
        ciphertext = f.read().strip()
    ciphertext = ''.join(c for c in ciphertext if c.isalpha()).upper()
    
    original_plaintext = ""
    if os.path.exists(plaintext_file):
        with open(plaintext_file, 'r', encoding='utf-8') as f:
            original_plaintext = f.read().strip()
        original_plaintext = ''.join(c for c in original_plaintext if c.isalpha()).upper()
    
    print(f"Ciphertext length: {len(ciphertext)} characters\n")
    
    # CRIBS - 
    cribs = [
        "ICHOSETHECONCERNS",        
    ]
    
    print("Cribs to search for:")
    for i, crib in enumerate(cribs, 1):
        print(f"  {i}. '{crib}' ({len(crib)} letters)")
    print()
    
###################################################################################################################################################
    bombe = Bombe()
    result = bombe.break_with_sliding_cribs(
        ciphertext=ciphertext,
        cribs=cribs,
        max_positions=17576, #basically this is all of the rotor positions = 26^3
        max_slide_distance=40,  # Search first until whatever point you choose where each "point" is up to a certain character in the text
        verbose=True
    )
###################################################################################################################################################
    
    if result and result.get('success', True):
        print("\n" + "="*80)
        print("✓ SUCCESS!")
        print("="*80)
        
        decrypted = decrypt_message(ciphertext, result['rotors'], result['positions'])
        accuracy = calculate_accuracy(decrypted, original_plaintext) if original_plaintext else 0.0
        
        save_results(result, decrypted, ciphertext, original_plaintext, 
                    accuracy, actual_settings, output_file)
        
        print(f"\nSettings: {result['rotors']} @ {result['positions_letters']}")
        print(f"Accuracy: {accuracy:.2f}%")
        print(f"Time: {result['time_seconds']:.2f}s")
        print("="*80 + "\n")
    else:
        print("\n✗ Failed to find solution")


if __name__ == "__main__":
    main()



BOMBE WITH SLIDING CRIB SEARCH

Ciphertext length: 2759 characters

Cribs to search for:
  1. 'ICHOSETHECONCERNS' (17 letters)


SLIDING CRIB SEARCH WITH IOC VERIFICATION
Cribs to try: 1
Searching first 40 characters
IoC threshold: 0.055  (English=0.065, Random=0.038)


CRIB 1/1: 'ICHOSETHECONCERNS' (17 letters)
Searching positions 0 to 40...

  Checked 20/40 positions...
  Crib match at text pos 21, rotor state (23, 0, 19) (XAT)
  Recovering initial positions from pos 21... Found: (2, 25, 19) (CZT)
  Verifying with initial pos (2, 25, 19) -> IoC: 0.0669  VERIFIED!

SOLUTION FOUND AND VERIFIED!
Crib 'ICHOSETHECONCERNS' at position:    21
Rotor config:                ('I', 'II', 'III')
Positions at crib:           (23, 0, 19) (XAT)
INITIAL positions (use this): (2, 25, 19) (CZT)
IoC Score:                   0.0669  (English=0.065)
Total tests:                 2,230,144
Time:                        104.89 seconds


✓ SUCCESS!
✓ Results saved to: bombe_results.txt

Settings: ('I', 'II', 